# Part 1: Self-Supervised Pretraining (BERT)

## Overview
This notebook demonstrates the **Self-Supervised Learning (SSL)** phase of our project. 
We train a **BERT-based Encoder** using a **Contrastive Loss (NT-Xent)** objective to learn robust packet representations from unlabeled network traffic.

### Goal
Train an encoder $f(x)$ such that augmented views of the same flow are close in the embedding space, while different flows are far apart. This learned representation is then transferred to downstream tasks (Anomaly Detection).

### Key Components
1.  **Data Loading**: Unlabeled UNSW-NB15 flows.
2.  **Augmentation**: Fast packet-level augmentation (Masking, Shuffle).
3.  **Model**: BERT Encoder with CLS Token Pooling.
4.  **Training**: Optimizing NT-Xent Loss.

In [1]:
import os, sys, pickle, time, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

print(f"Torch Version: {torch.__version__}")

Torch Version: 2.5.1+cu124


## Configuration
Define hyperparameters and file paths. Adjust `EPOCHS` for longer training.

In [2]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 128
EPOCHS = 1  # DEMO RUN (Increase to 50 for full reproduction)
LR = 1e-4
DATA_FILE = "../Organized_Final/data/unswnb15_full/finetune_mixed.pkl"
OUT_DIR = "weights/ssl"
os.makedirs(OUT_DIR, exist_ok=True)

print(f"Device: {DEVICE}")
print(f"Data: {DATA_FILE}")

Device: cuda
Data: ../Organized_Final/data/unswnb15_full/finetune_mixed.pkl


## Data Loading
We load the pre-processed flow data. Each flow consists of specific packet features (Protocol, Length, Flags, IAT, Direction).

In [3]:
print("Loading Data...")
with open(DATA_FILE, 'rb') as f: unsw_data = pickle.load(f)
# Use subset for demo speed if needed, or full data
unsw_train = unsw_data[:200000] 
X_train = np.array([d['features'] for d in unsw_train], dtype=np.float32)
print(f"Training Samples: {len(X_train)}")

Loading Data...


Training Samples: 200000


## Data Augmentation
Self-supervised learning requires two views of the same input. We apply random masking and shuffling to packets to generate these views.

In [4]:
def augment_flow(x):
    # x shape: (seq_len, features)
    mask = np.random.rand(x.shape[0]) < 0.15
    x_aug = x.copy()
    x_aug[mask] = 0 # Masking
    return x_aug

class  NTXentDataset(Dataset):
    def __init__(self, features):
        self.features = features
    def __len__(self):
        return len(self.features)
    def __getitem__(self, idx):
        x = self.features[idx]
        x_i = augment_flow(x)
        x_j = augment_flow(x)
        return torch.from_numpy(x_i), torch.from_numpy(x_j)

dl = DataLoader(NTXentDataset(X_train), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

## Model Architecture (BERT)
We use a Transformer Encoder (BERT-style). Key features:
- **Packet Embeddings**: Protocol, Flags, etc. are embedded and fused.
- **Transformer Layers**: 2-4 Layers of Self-Attention.
- **Pooling**: We use the **CLS Token** (First token) output as the sequence representation.
- **Projection Head**: Maps the representation to the contrastive space.

In [5]:
class BertEncoder(nn.Module):
    def __init__(self, d_model=256, nhead=8, num_layers=2):
        super().__init__()
        self.emb_proto = nn.Embedding(256, 16)
        self.emb_flags = nn.Embedding(64, 16)
        self.emb_dir   = nn.Embedding(2, 4)
        self.proj_len  = nn.Linear(1, 16)
        self.proj_iat  = nn.Linear(1, 16)
        self.fusion    = nn.Linear(68, d_model)
        self.norm      = nn.LayerNorm(d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=1024, dropout=0.1, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.proj_head = nn.Sequential(nn.Linear(d_model, d_model), nn.ReLU(), nn.Linear(d_model, 128))
    def forward(self, x):
        proto  = x[:,:,0].long().clamp(0, 255)
        length = x[:,:,1:2]
        flags  = x[:,:,2].long().clamp(0, 63)
        iat    = x[:,:,3:4]
        direc  = x[:,:,4].long().clamp(0, 1)
        cat = torch.cat([self.emb_proto(proto), self.proj_len(length),
                         self.emb_flags(flags), self.proj_iat(iat),
                         self.emb_dir(direc)], dim=-1)
        feat = self.norm(self.fusion(cat))
        feat = self.transformer_encoder(feat)
        # CLS Pooling
        return self.proj_head(feat[:, 0, :]), None

model = BertEncoder().to(DEVICE)
print("Model Initialized.")

Model Initialized.


## Contrastive Loss (NT-Xent)
We maximize representational similarity between views of the same flow.

In [6]:
class NTXentLoss(nn.Module):
    def __init__(self, temperature=0.5):
        super().__init__()
        self.temperature = temperature
        self.criterion = nn.CrossEntropyLoss()
    def forward(self, z_i, z_j):
        batch_size = z_i.shape[0]
        z = torch.cat([z_i, z_j], dim=0)
        sim = F.cosine_similarity(z.unsqueeze(1), z.unsqueeze(0), dim=2) / self.temperature
        sim_ij = torch.diag(sim, batch_size)
        sim_ji = torch.diag(sim, -batch_size)
        positives = torch.cat([sim_ij, sim_ji], dim=0)
        mask = (~torch.eye(2 * batch_size, 2 * batch_size, dtype=bool)).to(z.device)
        negatives = sim[mask].view(2 * batch_size, -1)
        logits = torch.cat([positives.unsqueeze(1), negatives], dim=1)
        labels = torch.zeros(2 * batch_size, dtype=torch.long).to(z.device)
        return self.criterion(logits, labels)

criterion = NTXentLoss().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

## Training Loop
We iterate through the dataset, optimizing the contrastive loss.

In [7]:
print(f"Starting Training for {EPOCHS} Epoch(s)...")
model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    start = time.time()
    for batch_idx, (x_i, x_j) in enumerate(dl):
        x_i, x_j = x_i.to(DEVICE), x_j.to(DEVICE)
        optimizer.zero_grad()
        z_i, _ = model(x_i)
        z_j, _ = model(x_j)
        loss = criterion(z_i, z_j)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        if batch_idx % 100 == 0:
             print(f"Epoch {epoch+1} [{batch_idx}/{len(dl)}] Loss: {loss.item():.4f}")
    print(f"Epoch {epoch+1} Complete. Avg Loss: {total_loss/len(dl):.4f}. Time: {time.time()-start:.0f}s")

# Save Weights
save_path = f"{OUT_DIR}/bert_standard_ssl_optimized.pth"
torch.save(model.state_dict(), save_path)
print(f"Weights saved to {save_path}")

Starting Training for 1 Epoch(s)...


Epoch 1 [0/1562] Loss: 5.5076


Epoch 1 [100/1562] Loss: 4.5119


Epoch 1 [200/1562] Loss: 4.2287


Epoch 1 [300/1562] Loss: 4.1363


...

Epoch 1 [1200/1562] Loss: 3.9655


Epoch 1 [1300/1562] Loss: 4.0873


Epoch 1 [1400/1562] Loss: 4.1091


Epoch 1 [1500/1562] Loss: 3.9956


Epoch 1 Complete. Avg Loss: 4.1126. Time: 19s
Weights saved to weights/ssl/bert_standard_ssl_optimized.pth
